# Smart Fog Vision — Supervised Fog Removal Model Training
### Dataset: `mohammedawais505/supervised-fog-removal-dataset` (80K Pairs)

This notebook trains a physics-inspired deep learning dehazing network (**AOD-Net**) directly on Kaggle with **Zero Local Downloads** and **Free GPU Acceleration**.

---
### How to Run on Kaggle (1-Click, Zero Download):
1. Open [https://www.kaggle.com/datasets/mohammedawais505/supervised-fog-removal-dataset](https://www.kaggle.com/datasets/mohammedawais505/supervised-fog-removal-dataset)
2. Click the blue **'New Notebook'** button at top right.
3. In the right panel under **Settings**, set **Accelerator → GPU T4 x2 or P100** and enable **Internet → On**.
4. Click **File → Import Notebook** and upload this file (`Supervised_Fog_Removal_Kaggle_Trainer.ipynb`).
5. Click **Run All**.
6. In the right panel under **Output** (`/kaggle/working/`), download `fog_dehazer.onnx` and `best_fog_dehazer.pt` and place them into `Smart_Fog_Vision/models/`!

In [ ]:
import os
import glob
import time
from pathlib import Path

# Check Kaggle environment & dataset mounting
print("Checking Kaggle input directories:")
input_dirs = glob.glob('/kaggle/input/**/*', recursive=False)
print("Mounted inputs:", input_dirs)

dataset_root = '/kaggle/input/supervised-fog-removal-dataset'
if os.path.exists(dataset_root):
    print(f"Dataset mounted at {dataset_root}. Contents:", os.listdir(dataset_root)[:10])
else:
    print("Running outside Kaggle. Will fallback to online streaming via Hugging Face.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Model Definition: AOD-Net (All-in-One Dehazing Network)
class AODNet(nn.Module):
    """
    AOD-Net (Boyi Li et al., ICCV 2017)
    Reformulates atmospheric scattering: J(x) = K(x) * I(x) - K(x) + b
    Compact (~2,000 parameters) — perfect for high-speed 60+ FPS web inference.
    """
    def __init__(self):
        super(AODNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 3, kernel_size=1)
        self.conv2 = nn.Conv2d(3, 3, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(6, 3, kernel_size=5, padding=2)
        self.conv4 = nn.Conv2d(6, 3, kernel_size=7, padding=3)
        self.conv5 = nn.Conv2d(12, 3, kernel_size=3, padding=1)
        self.b = 1.0

    def forward(self, x):
        x1 = F.relu(self.conv1(x))
        x2 = F.relu(self.conv2(x1))
        cat1 = torch.cat((x1, x2), dim=1)
        x3 = F.relu(self.conv3(cat1))
        cat2 = torch.cat((x2, x3), dim=1)
        x4 = F.relu(self.conv4(cat2))
        cat3 = torch.cat((x1, x2, x3, x4), dim=1)
        k = F.relu(self.conv5(cat3))
        
        output = k * x - k + self.b
        output = F.relu(output)
        return torch.clamp(output, 0.0, 1.0)

model = AODNet().to(device)
print(f"AOD-Net instantiated. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Dataset Loading (Kaggle Mounted Files or Online Streaming)
import subprocess
try:
    import datasets
except ImportError:
    subprocess.run(["pip", "install", "datasets", "-q"])
    import datasets
from datasets import load_dataset

IMG_SIZE = 256
BATCH_SIZE = 32 if torch.cuda.is_available() else 8
TRAIN_SAMPLES = 2000  # Number of pairs to train on (increase to 10000+ for full training)
VAL_SAMPLES = 200

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor()
])

class StreamDehazeDataset(torch.utils.data.IterableDataset):
    def __init__(self, split='train', max_samples=1000):
        self.split = split
        self.max_samples = max_samples
    def __iter__(self):
        ds = load_dataset('Aeye-coder/Supervised-Fog-Removal-Dataset', split=self.split, streaming=True)
        count = 0
        for sample in ds:
            if count >= self.max_samples: break
            try:
                clean = transform(sample['clean'].convert('RGB'))
                foggy = transform(sample['foggy'].convert('RGB'))
                yield foggy, clean
                count += 1
            except Exception:
                continue

train_ds = StreamDehazeDataset('train', max_samples=TRAIN_SAMPLES)
val_ds = StreamDehazeDataset('validation', max_samples=VAL_SAMPLES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
print(f"DataLoaders initialized (Batch size: {BATCH_SIZE}, Device: {device})")

In [ ]:
# Training Configuration & Execution Loop
EPOCHS = 5
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float('inf')
os.makedirs('/kaggle/working', exist_ok=True)

print("Starting Training...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss, train_b = 0.0, 0
    
    for i, (foggy, clean) in enumerate(train_loader):
        foggy, clean = foggy.to(device), clean.to(device)
        optimizer.zero_grad()
        pred = model(foggy)
        loss = criterion(pred, clean)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_b += 1
        if (i + 1) % 15 == 0:
            print(f"  Epoch [{epoch}/{EPOCHS}] | Step {i+1} | Loss: {loss.item():.4f}")
            
    scheduler.step()
    avg_train_loss = train_loss / max(1, train_b)
    
    # Validation
    model.eval()
    val_loss, val_b = 0.0, 0
    val_psnr_list = []
    with torch.no_grad():
        for foggy, clean in val_loader:
            foggy, clean = foggy.to(device), clean.to(device)
            pred = model(foggy)
            loss = criterion(pred, clean)
            val_loss += loss.item()
            mse = F.mse_loss(pred, clean).item()
            psnr = 10 * np.log10(1.0 / max(mse, 1e-8))
            val_psnr_list.append(psnr)
            val_b += 1
            
    avg_val_loss = val_loss / max(1, val_b)
    avg_psnr = np.mean(val_psnr_list) if val_psnr_list else 0.0
    elapsed = time.time() - t0
    
    print(f"Epoch {epoch}/{EPOCHS} [{elapsed:.1f}s] -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val PSNR: {avg_psnr:.2f} dB")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), '/kaggle/working/best_fog_dehazer.pt')
        print("  [*] Saved new best model checkpoint!")

torch.save(model.state_dict(), '/kaggle/working/fog_dehazer.pt')
print("Training Complete!")

In [ ]:
# Export to ONNX (Browser-ready dynamic shape for ONNX Runtime Web)
model.eval()
onnx_path = '/kaggle/working/fog_dehazer.onnx'
dummy_in = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)

torch.onnx.export(
    model,
    dummy_in,
    onnx_path,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size', 2: 'height', 3: 'width'},
        'output': {0: 'batch_size', 2: 'height', 3: 'width'}
    }
)

print(f"Successfully exported ONNX: {onnx_path}")
print(f"ONNX File Size: {os.path.getsize(onnx_path) / 1024:.1f} KB")

In [ ]:
# Visual Demonstration: Compare Foggy Input vs Dehazed Output vs Ground Truth
model.eval()
with torch.no_grad():
    for foggy, clean in val_loader:
        foggy, clean = foggy.to(device), clean.to(device)
        dehazed = model(foggy)
        
        # Show first sample
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        axs[0].imshow(foggy[0].cpu().permute(1, 2, 0).clamp(0, 1).numpy())
        axs[0].set_title("Foggy Input (Synthetic Fog)")
        axs[0].axis('off')
        
        axs[1].imshow(dehazed[0].cpu().permute(1, 2, 0).clamp(0, 1).numpy())
        axs[1].set_title("AOD-Net Restored / Dehazed")
        axs[1].axis('off')
        
        axs[2].imshow(clean[0].cpu().permute(1, 2, 0).clamp(0, 1).numpy())
        axs[2].set_title("Ground Truth (Clean)")
        axs[2].axis('off')
        
        plt.tight_layout()
        plt.show()
        break

In [ ]:
# Verification of Generated Output Files ready for download
print("Generated files ready for download in /kaggle/working/:")
for f in os.listdir('/kaggle/working'):
    if f.endswith(('.onnx', '.pt', '.json')):
        size_kb = os.path.getsize(os.path.join('/kaggle/working', f)) / 1024
        print(f"  -> {f} ({size_kb:.1f} KB)")
print("\nDone! In Kaggle's right sidebar under 'Output', hover over fog_dehazer.onnx and click Download.")